# Compiling RxInfer with Reactant

[Reactant.jl](https://github.com/EnzymeAD/Reactant.jl) compiles Julia functions into [XLA](https://openxla.org/xla) programs for CPUs, GPUs and TPUs. It does so by tracing: the function runs once with traced arrays, which record every array operation, while everything else (control flow on plain values, dictionaries, dispatch) runs as ordinary Julia at trace time and leaves no trace in the compiled program.

RxInfer's inference is a reactive program: messages flow through subscriptions between the nodes of a factor graph. Under tracing, all of that machinery runs once, and what remains is the arithmetic of the messages. A call to `infer` therefore compiles into a single straight-line program.

This example compiles one step of a Kalman filter written as an RxInfer model, and runs it in a loop over a stream of observations, each posterior becoming the next prior.

!!! note
    This example needs features not yet released: traced floating-point numbers that are `Real` (EnzymeAD/Reactant.jl#2170), FastCholesky's Reactant extension (ReactiveBayes/FastCholesky.jl#39), and fixes in GraphPPL (ReactiveBayes/GraphPPL.jl#334) and RxInfer (ReactiveBayes/RxInfer.jl#702).

In [ ]:
using RxInfer, Reactant, MatrixCorrectionTools, LinearAlgebra, Random, StableRNGs, BenchmarkTools, Plots

## One step of the filter

The hidden state rotates by a fixed angle at every step and is observed with noise:

$$\begin{aligned}
x_t &\sim \mathcal{N}(A x_{t-1}, P), \\
y_t &\sim \mathcal{N}(B x_t, Q).
\end{aligned}$$

One step of filtering takes the belief about the previous state, $\mathcal{N}(m_{t-1}, V_{t-1})$, and one observation, and returns the belief about the current state. As an RxInfer model:

In [ ]:
@model function filter_step(y, m_prev, V_prev, A, B, P, Q)
    x_prev ~ MvNormalMeanCovariance(m_prev, V_prev)
    x ~ MvNormalMeanCovariance(A * x_prev, P)
    y ~ MvNormalMeanCovariance(B * x, Q)
end

The function we compile runs the inference and returns the mean and the covariance of the posterior of `x`. The multiplication node corrects a precision matrix with a zero on its diagonal by default, which needs to read the matrix's values; under tracing they are unknown, so the correction is switched off.

In [ ]:
function step(y, m_prev, V_prev, A, B, P, Q)
    result = infer(
        model = filter_step(m_prev = m_prev, V_prev = V_prev, A = A, B = B, P = P, Q = Q),
        data = (y = y,),
        options = (context = (matrix_correction = MatrixCorrectionTools.NoCorrection(),),),
    )
    posterior = result.posteriors[:x]
    return mean(posterior), cov(posterior)
end

## Data

In [ ]:
θ = π / 35
A = [cos(θ) -sin(θ); sin(θ) cos(θ)]
B = diageye(2)
P = diageye(2)
Q = 25.0 * diageye(2)

rng = StableRNG(1234)
n = 300
x = Vector{Vector{Float64}}(undef, n)
y = Vector{Vector{Float64}}(undef, n)
previous = zeros(2)
for t in 1:n
    x[t] = rand(rng, MvNormal(A * previous, P))
    y[t] = rand(rng, MvNormal(B * x[t], Q))
    global previous = x[t]
end

## Filtering with RxInfer

The plain filter runs `step` on ordinary arrays, which creates and runs a small factor graph at every step.

In [ ]:
function run_filter(step, y, m0, V0, A, B, P, Q)
    m, V = m0, V0
    means, covariances = [], []
    for yt in y
        m, V = step(yt, m, V, A, B, P, Q)
        push!(means, m)
        push!(covariances, V)
    end
    return means, covariances
end

m0, V0 = zeros(2), 100.0 * diageye(2)
means, covariances = run_filter(step, y, m0, V0, A, B, P, Q);

## Filtering with the compiled step

`Reactant.to_rarray` moves the arrays to the device. `@compile` traces `step` once with them and returns the compiled program, which then runs for every observation; the posterior it returns, already on the device, is the next step's prior.

In [ ]:
ry = [Reactant.to_rarray(yt) for yt in y]
rm0, rV0, rA, rB, rP, rQ = Reactant.to_rarray.((m0, V0, A, B, P, Q))

compile_time = @elapsed compiled_step = @compile sync = true step(first(ry), rm0, rV0, rA, rB, rP, rQ)
println("compiled in ", round(compile_time, digits = 1), " s")

compiled_means, compiled_covariances = run_filter(compiled_step, ry, rm0, rV0, rA, rB, rP, rQ);

The compiled filter computes the same posteriors:

In [ ]:
difference = maximum(zip(means, compiled_means, covariances, compiled_covariances)) do (m, rm, V, rV)
    return max(maximum(abs, m - Array(rm)), maximum(abs, V - Array(rV)))
end
println("largest difference: ", difference)

In [ ]:
filtered = Array.(compiled_means)
stds = [sqrt.(diag(Array(V))) for V in compiled_covariances]
p1 = plot(getindex.(x, 1), label = "hidden state", color = :black, ylabel = "first coordinate")
scatter!(p1, getindex.(y, 1), label = "observations", markersize = 2, alpha = 0.4)
plot!(p1, getindex.(filtered, 1), ribbon = getindex.(stds, 1), label = "filtered (compiled)")
p2 = plot(getindex.(x, 2), label = "hidden state", color = :black, ylabel = "second coordinate", xlabel = "time")
scatter!(p2, getindex.(y, 2), label = "observations", markersize = 2, alpha = 0.4)
plot!(p2, getindex.(filtered, 2), ribbon = getindex.(stds, 2), label = "filtered (compiled)")
plot(p1, p2, layout = (2, 1), size = (800, 600), legend = :bottomleft)

## What the compiled program contains

The traced program holds only the arithmetic of the messages. Its size:

In [ ]:
hlo = string(@code_hlo step(first(ry), rm0, rV0, rA, rB, rP, rQ))
println(count(==('\n'), hlo), " lines, ", length(collect(eachmatch(r"stablehlo\.\w+", hlo))), " StableHLO operations")

## Timings

One step with RxInfer creates a factor graph, subscribes its nodes to each other and propagates the messages; the compiled step runs only the arithmetic.

In [ ]:
plain_trial = @benchmark step($(y[1]), $m0, $V0, $A, $B, $P, $Q)
compiled_trial = @benchmark $compiled_step($(ry[1]), $rm0, $rV0, $rA, $rB, $rP, $rQ)
for (name, trial) in (("RxInfer", plain_trial), ("compiled", compiled_trial))
    estimate = median(trial)
    println(rpad(name, 10), BenchmarkTools.prettytime(estimate.time), " per step, ", estimate.allocs, " allocations")
end
println("speed-up: ", round(median(plain_trial).time / median(compiled_trial).time, digits = 1), "×")

In [ ]:
plain_total = @elapsed run_filter(step, y, m0, V0, A, B, P, Q)
compiled_total = @elapsed run_filter(compiled_step, ry, rm0, rV0, rA, rB, rP, rQ)
bar(
    ["RxInfer", "compiled"], 1e3 .* [plain_total, compiled_total],
    ylabel = "milliseconds for $(n) steps", legend = false, color = [:gray, :seagreen],
    title = "Filtering $(n) observations",
)

## Larger states

The same step, for states of more dimensions: the rotation becomes a random orthogonal matrix, and every dimension is observed. RxInfer's cost per step is mostly the graph and the engine, so it grows slowly with the dimension at first; the compiled step is only the arithmetic, so it grows with the cost of the matrix operations.

Factorisations of traced matrices go through LU in FastCholesky: on CPU, Reactant's Cholesky factorisation is currently much slower than LAPACK's, while its LU factorisation calls LAPACK (EnzymeAD/Reactant.jl#3411).

In [ ]:
function random_problem(d; rng = StableRNG(d))
    A = Matrix(qr(randn(rng, d, d)).Q)
    B, P, Q = diageye(d), diageye(d), 25.0 * diageye(d)
    y = rand(rng, MvNormal(A * randn(rng, d), Q))
    return y, zeros(d), 100.0 * diageye(d), A, B, P, Q
end

dimensions = [2, 4, 8, 16, 32, 64]
plain_times, compiled_times, compile_times = Float64[], Float64[], Float64[]
for d in dimensions
    args = random_problem(d)
    rargs = Reactant.to_rarray.(args)
    push!(compile_times, @elapsed(compiled = @compile sync = true step(rargs...)))
    @assert maximum(abs, first(step(args...)) - Array(first(compiled(rargs...)))) < 1e-8
    push!(plain_times, @belapsed step($(args)...))
    push!(compiled_times, @belapsed $compiled($(rargs)...))
end

println("    d     RxInfer    compiled  speed-up  compile")
for (d, p, c, t) in zip(dimensions, plain_times, compiled_times, compile_times)
    println(
        lpad(d, 5), lpad(BenchmarkTools.prettytime(1e9 * p), 12), lpad(BenchmarkTools.prettytime(1e9 * c), 12),
        lpad(string(round(p / c, digits = 1), "×"), 10), lpad(string(round(t, digits = 1), " s"), 9),
    )
end

In [ ]:
plot(
    dimensions, 1e6 .* [plain_times compiled_times],
    label = ["RxInfer" "compiled"], marker = :circle, xscale = :log2, yscale = :log10,
    xticks = (dimensions, string.(dimensions)), xlabel = "dimension of the state", ylabel = "μs per step",
    title = "One filtering step", legend = :topleft,
)

Compiling takes a while, once: it pays off when the same step runs many times, as in a filter over a long or endless stream of observations. The program is fixed for the shapes it was compiled with; a model of a different size needs its own compilation.

The same program also runs on a GPU, with `Reactant.set_default_backend("gpu")` on a machine with an NVIDIA or AMD GPU (Linux only); the larger the state, the more a GPU has to work with.